# Benchmark models and ablations


In [ ]:
import json
import os
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import contextmanager

PACKAGE_ROOT = Path.cwd()
if not (PACKAGE_ROOT / "geoctgan").is_dir():
    PACKAGE_ROOT = PACKAGE_ROOT.parent
if not (PACKAGE_ROOT / "geoctgan").is_dir():
    raise FileNotFoundError("geoctgan")
os.chdir(PACKAGE_ROOT)
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

from geoctgan.config import Config

def input_path(label, required=True):
    raw = input(label + ": ").strip().strip('"').strip("'")
    if not raw:
        if not required:
            return None
        raise ValueError(label)
    path = Path(raw).expanduser().resolve()
    if required and not path.exists():
        raise FileNotFoundError(label)
    return path

def stored_path(path):
    return str(Path(path).resolve())

@contextmanager
def configuration_file(values):
    with TemporaryDirectory(prefix="geoctgan_") as directory:
        path = Path(directory) / "configuration.json"
        path.write_text(json.dumps(values, indent=2) + "\n", encoding="utf-8")
        yield path


## Generation inputs


In [ ]:
from geoctgan.evaluation import resolve_recorded_artifact
from geoctgan.utils import load_output_json

GENERATION_SUMMARY = input_path("Generation run_summary.json")
generation = load_output_json(GENERATION_SUMMARY)
MODEL_SEEDS = [42, 52, 62]


## GaussianCopula, raw-coordinate CTGAN, KDE-AttributeBootstrap and RoadSupportBootstrap


In [ ]:
from geoctgan.benchmarks import run_benchmark_generation
from geoctgan.evaluation import resolve_recorded_artifact
config_path = resolve_recorded_artifact(generation["config"], manifest_path=GENERATION_SUMMARY, label="config")
benchmark_cfg = Config.from_json(str(config_path))
benchmark_cfg.BENCHMARK_SEEDS = MODEL_SEEDS
benchmark_cfg.BENCHMARK_METHODS = ["GaussianCopula", "CTGAN_RawLatLon", "KDE_AttributeBootstrap", "RoadSupportBootstrap"]
snapped_train = resolve_recorded_artifact(generation["snapped_train"], manifest_path=GENERATION_SUMMARY, label="snapped_train")
BENCHMARK_MANIFEST = run_benchmark_generation(benchmark_cfg, str(snapped_train), str(GENERATION_SUMMARY.parent.parent))
